### Test

In [ ]:
sorted_items = sorted(
    ((k, v) for k, v in hash_table_creation.items() if len(v[1]) > 1),
    key=lambda item: len(item[1][1])
)

# Print the key along with its associated name and children
for k, v in sorted_items:
    print(f"{k}: {v[0]} -> {v[1]}")

### Main Script

In [ ]:
import pandas as pd

In [ ]:
def process_csv(file_path):
    df = pd.read_csv(file_path)
    filtered_df = df[df["ProviderName_0"] == "Microsoft-Windows-Sysmon"]

    # Marked columns
    priority_cols = [

        "EventID_0",
        "TimeCreated_0",

        "ProcessGuid_2",
        "ProcessId_2",
        "ParentProcessGuid_2",
        "ParentProcessId_2",


        "SourceProcessGuid_2",
        "SourceProcessId_2",
        "TargetProcessGuid_2",
        "TargetProcessId_2"
    ]

    # Keep priority cols first, then the rest
    remaining_cols = [c for c in filtered_df.columns if c not in priority_cols]
    ordered_df = filtered_df[priority_cols + remaining_cols]


    return ordered_df




In [ ]:

file_path = "multi_log_events_bigger.csv"
result_df = process_csv(file_path)

hash_table_creation = {}
hash_table_source_target = {}



# the code below this actually creates the tree to be traversed from the input csv file.
for _, row in result_df.iterrows():
    process_guid = row["ProcessGuid_2"]
    parent_guid = row["ParentProcessGuid_2"]
    event_id = row["EventID_0"]
    time_created = row["TimeCreated_0"]
    process_id = row["ProcessId_2"]
    parent_id = row["ParentProcessId_2"]


    source_guid = row["SourceProcessGuid_2"]
    source_id = row["SourceProcessId_2"]
    target_guid = row["TargetProcessGuid_2"]
    target_id = row["TargetProcessId_2"]

    process_valid = pd.notna(process_guid)
    process_in = process_guid in hash_table_creation



    parent_valid = pd.notna(parent_guid)
    parent_in = parent_guid in hash_table_creation

    # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
    #     print(f"\nProcess GUID: {process_guid}, process_in: {process_in}, Event ID: {event_id}, Process_valid:{process_valid}")

    if process_valid:
        if event_id  ==  1:
            # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
            #     print(f"\n 2 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")


            if not process_in and process_valid:
                hash_table_creation[process_guid] = [[time_created, process_id], [], [parent_guid]]
                # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #     print(f"\n 3 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")
                #     print(hash_table_creation[process_guid])
            elif process_in and process_valid:
                curr_time_created = hash_table_creation[process_guid][0][0]
                hash_table_creation[process_guid][0] = ([max(curr_time_created, time_created), process_id])
                parent_list = hash_table_creation[process_guid][2]
                if parent_guid not in parent_list:
                    parent_list.append(parent_guid)
                    hash_table_creation[process_guid][2] = parent_list
                #     if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #         print("got appended")
                #         print(hash_table_creation[process_guid][2])
                # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #     print(f"\n 3.5 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")
                #     print(hash_table_creation[process_guid])

            if not parent_in and parent_valid:
                hash_table_creation[parent_guid] = [[time_created, parent_id], [process_guid], []]
                # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #     print(f"\n 4 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")
                #     print(hash_table_creation[parent_guid])

            elif parent_in and parent_valid:
                curr_time_created = hash_table_creation[parent_guid][0][0]
                hash_table_creation[parent_guid][0] = ([max(curr_time_created, time_created), parent_id])
                process_list = hash_table_creation[parent_guid][1]
                if process_guid not in process_list:
                    process_list.append(process_guid)
                    hash_table_creation[parent_guid][1] = process_list
                # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #     print(f"\n 5 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")
                #     print(hash_table_creation[parent_guid])

        else:
            if not process_in and process_valid:
                hash_table_creation[process_guid] = [[time_created, process_id], [], []]
                # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #     print(f"\n 6 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")
                #     print(hash_table_creation[process_guid])

            elif process_in and process_valid:
                curr_time_created = hash_table_creation[process_guid][0][0]
                hash_table_creation[process_guid][0] = ([max(curr_time_created, time_created), process_id])
                # if process_guid == "{65c548b6-1af9-68c9-e916-000000001500}":
                #     print(f"\n 7 Event ID: {event_id}, Process GUID: {process_guid}, Parent GUID: {parent_guid}")
                #     print(hash_table_creation[process_guid])



    if event_id in (8, 10):
        if source_guid in hash_table_source_target:
            curr_time = hash_table_source_target[source_guid][0][0]
            hash_table_source_target[source_guid][0] = ([max(curr_time, time_created), target_id])

            if target_guid not in hash_table_source_target[source_guid][1]:
                hash_table_source_target[source_guid][1].append(target_guid)


        else:
            hash_table_source_target[source_guid] = [[time_created, source_id], [target_guid], []]

        if target_guid in hash_table_source_target:
            curr_time = hash_table_source_target[target_guid][0][0]
            hash_table_source_target[target_guid][0] = ([max(curr_time, time_created), source_id])

            if source_guid not in hash_table_source_target[target_guid][2]:
                hash_table_source_target[target_guid][2].append(source_guid)

        else:
            hash_table_source_target[target_guid] = [[time_created, target_id], [], [source_guid]]

# print("\nHash Table:", hash_table_creation)

/tmp/ipython-input-94954317.py:2: DtypeWarning: Columns (14,17,18,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,36,37,38,39,40,41,42,43,44,45,46,48,49,50,51,53,57,58,59,60,61,63,64,65,70,72,73,74,75,76,77,82,90,93,94,95,98,102,106,107,108,117,119) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [ ]:
# ----------------------------
# Helper Functions
# ----------------------------
def remove_redundant_data(log: str) -> str:
    return log

def get_creation_log(pid: str) -> str:
    row = result_df[
        (result_df["ProviderName_0"] == "Microsoft-Windows-Sysmon") &
        (result_df["EventID_0"] == 1) &
        (result_df["ProcessGuid_2"] == pid)
    ]
    if row.empty:
        return ""
    row = row.iloc[0]
    event_data = str(row["EventData_1"]).replace("\n", ", ")
    system_data = str(row["SystemData_1"]).replace("\n", ", ")
    return (event_data + ", " + system_data).strip()

def get_actions(pid: str) -> list[str]:
    sysmon_event_map = {
        2: "File creation time changed", 3: "Network connection detected", 5: "Process terminated",
        7: "Image loaded", 9: "Security event", 11: "File create", 12: "Registry event",
        13: "Registry value set", 14: "Registry key deleted", 17: "Pipe event", 18: "WmiEvent", 22: "DnsQuery"
    }
    rows = result_df[
        (result_df["ProviderName_0"] == "Microsoft-Windows-Sysmon") &
        (result_df["EventID_0"].isin(sysmon_event_map.keys())) &
        (result_df["ProcessGuid_2"] == pid)
    ].copy()
    if rows.empty:
        return []
    rows = rows.sort_values("TimeCreated_0")
    actions_list = []
    for _, row in rows.iterrows():
        event_id = int(row["EventID_0"])
        description = sysmon_event_map.get(event_id, f"Sysmon Event {event_id}")
        event_data = str(row.get("EventData_1", "")).replace("\n", ", ")
        system_data = str(row.get("SystemData_1", "")).replace("\n", ", ")
        user_data = str(row.get("UserData_1", "")).replace("\n", ", ")
        combined = f"{description}: {event_data}, {system_data}, {user_data}".strip(", ")
        actions_list.append(combined)
    return actions_list

def get_children(parent_pid: str, exclude_pid: str = None) -> list[str]:
    # Filter creation events where this process is the parent
    rows = result_df[
        (result_df["ProviderName_0"] == "Microsoft-Windows-Sysmon") &
        (result_df["EventID_0"] == 1) &
        (result_df["ParentProcessGuid_2"] == parent_pid)
    ].copy()

    if rows.empty:
        return []

    # Sort by creation time
    rows = rows.sort_values("TimeCreated_0")

    # Extract all child GUIDs
    children = rows["ProcessGuid_2"].tolist()

    # Remove only the process in the main chain
    if exclude_pid and exclude_pid in children:
        children.remove(exclude_pid)

    return children


def get_parent_guid(process_guid: str, tree: dict) -> str:
    if process_guid not in tree:
        return None
    parent_list = tree[process_guid][2]
    if not parent_list:
        return None
    return parent_list[0]

def get_ancestors(process_guid: str, n: int, tree: dict) -> tuple[list[str], int]:
    ancestors = [process_guid]
    counter = 0
    current_guid = process_guid
    while counter < n:
        parent = get_parent_guid(current_guid, tree)
        if not parent:
            break
        ancestors.append(parent)
        current_guid = parent
        counter += 1
    ancestors.reverse()
    return ancestors, len(ancestors)

# ----------------------------
# Build LLM-ready nested context tree
# ----------------------------
def build_process_context_json(parent_chain: list[str]) -> dict:
    context_json = {}
    n = len(parent_chain)
    for i, pid in enumerate(parent_chain):
        k = n - i
        creation_log = get_creation_log(pid)
        details = remove_redundant_data(creation_log)
        node_label = f"Level-{k} Process ({pid})"
        node_entry = {"Details": details}
        if k > 3:
            context_json[node_label] = node_entry
            continue
        actions_list = get_actions(pid)
        node_entry["Actions"] = remove_redundant_data("\n".join(actions_list))
        if k in [3, 2]:
            next_pid = parent_chain[i + 1] if i + 1 < n else None
            children_pids = get_children(pid, exclude_pid=next_pid)
            child_entries = {}
            for child_pid in children_pids:
                child_details = remove_redundant_data(get_creation_log(child_pid))
                child_actions = remove_redundant_data("\n".join(get_actions(child_pid)))
                child_entries[f"Child Process ({child_pid})"] = {
                    "Details": child_details,
                    "Actions": child_actions
                }
            if child_entries:
                node_entry["Children"] = child_entries
        context_json[node_label] = node_entry
    return context_json

In [ ]:
parent_chain, _ = get_ancestors("{65c548b6-1add-68c9-e116-000000001500}", n=5, tree=hash_table_creation)
context_tree = build_process_context_json(parent_chain)

import json
print(json.dumps(context_tree, indent=4))

{
    "Level-2 Process ({65c548b6-75ac-68c1-0b00-000000001500})": {
        "Details": "",
        "Actions": "Registry value set: RuleName: -, EventType: SetValue, UtcTime: 2025-09-15 10:41:59.789, ProcessGuid: {65c548b6-75ac-68c1-0b00-000000001500}, ProcessId: 704, Image: C:\\Windows\\system32\\services.exe, TargetObject: HKLM\\System\\CurrentControlSet\\Services\\BITS\\Start, Details: DWORD (0x00000002), User: NT AUTHORITY\\SYSTEM, Provider.Name: Microsoft-Windows-Sysmon, Provider.Guid: {5770385f-c22a-43e0-bf4c-06f5698ffbd9}, EventID: 13, Version: 2, Level: 4, Task: 13, Opcode: 0, Keywords: 0x8000000000000000, TimeCreated.SystemTime: 2025-09-15T10:41:59.7973303Z, EventRecordID: 2165748, Execution.ProcessID: 3412, Execution.ThreadID: 4036, Channel: Microsoft-Windows-Sysmon/Operational, Computer: DESKTOP-IAE5LS4, Security.UserID: S-1-5-18, nan\nRegistry value set: RuleName: -, EventType: SetValue, UtcTime: 2025-09-15 10:44:05.838, ProcessGuid: {65c548b6-75ac-68c1-0b00-000000001500}, P